In [ ]:
from glob import glob
import os
import pandas as pd
import json
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import warnings ; warnings.filterwarnings('ignore')
import numpy as np

import torch
from transformers import AutoModel, AutoTokenizer
from src.utils.preprocessing import Preprocessor

from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib import font_manager as fm

from umap import UMAP
from scipy.stats import f
from tqdm import tqdm

def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df


font_fname = os.environ.get('PROJECT_FONT_PATH', '') # point at any local CJK/serif .ttf
font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용



negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]


target_class = ['negative','uncertain','positive']
cancer_merge_keys = ['Neurologic', 'Hematologic', 'Head & Neck', 'Endocrine']
# merged_cancer_label = 'Neurologic & Hematologic & HeadNeck & Endocrine'
merged_cancer_label = 'Merged'

figdir = os.path.join('.','figure')
os.makedirs(figdir, exist_ok = True)

In [ ]:

rev_dir_path = os.path.join('data','reviewer')
metas_df, recur_df = [pd.read_excel(i, sheet_name=None) for i in sorted(glob(os.path.join(rev_dir_path,'*.xlsx')))]


In [ ]:
agg_df_save_dir = os.path.join('data','reviewed_labels')
load_all = pd.read_excel([i for i in glob(os.path.join(agg_df_save_dir, '*')) if '.xlsx' in i][1])
# cancer_type_df = pd.read_excel('data/cancer_type_map.xlsx')


In [ ]:
load_all['bMetastasis'] = load_all['Metastasis'].apply(lambda x: 'positive' if x == 'positive' else 'negative')
load_all['bRecurrence'] = load_all['Recurrence'].apply(lambda x: 'positive' if x == 'positive' else 'negative')

In [ ]:
import pandas as pd

stats_dict = dict()
# 필요하다면 non_conflict 데이터를 저장할 딕셔너리도 추가할 수 있습니다.
# non_conflict_dict = dict() 

cols_to_group = ['병원등록번호', '수술일자', '검사접수일자']

for target in ['Metastasis', 'Recurrence']:
    target_col = 'b' + target
    
    # 1. 기본 데이터 준비 (중복 제거)
    subset_df = load_all[['병원등록번호','성별','암종','수술일자','검사접수일자', target_col]].drop_duplicates()
    
    # 2. 충돌 여부 마스킹 (그룹 내 값의 종류가 1개 초과인지 확인)
    # True: 충돌 있음(섞임), False: 충돌 없음(일관됨)
    has_conflict = subset_df.groupby(cols_to_group)[target_col].transform('nunique') > 1
    
    # 3. 데이터 분리 (요청하신 부분)
    conflict = subset_df[has_conflict].copy()       # 값이 섞여 있는 데이터 (처리 대상)
    non_conflict = subset_df[~has_conflict].copy()  # 값이 일관되거나 하나인 데이터 (유지 대상)
    
    # # 4. Conflict 데이터 처리 로직 (무조건 Positive로 병합)
    # if not conflict.empty:
    # 그룹별 첫 번째 행만 남김
    conflict_processed = conflict.drop_duplicates(subset=cols_to_group, keep='first').copy()
    # 타겟 값을 'positive'로 강제 설정
    conflict_processed[target_col] = 'positive'
    
    # 결과 저장
    stats_dict[target] = pd.concat([non_conflict,conflict_processed])
    
    # 수술일자와 검사접수일자를 datetime으로 변환 (형태: YYYYMMDD → datetime)
    temp_df = stats_dict[target].copy()
    for col in ['수술일자', '검사접수일자']:
        temp_df[col] = pd.to_datetime(temp_df[col], format='%Y%m%d', errors='coerce')
        
    # 검사접수일자 - 수술일자(일 수 차이 계산)
    temp_df['Terms'] = (
        pd.to_datetime(temp_df['검사접수일자'], format='%Y%m%d', errors='coerce') - \
        pd.to_datetime(temp_df['수술일자'], format='%Y%m%d', errors='coerce')
    ).dt.days

    # 원래 sort_index도 필요하면 추가로 적용
    temp_df = temp_df.sort_index()

    stats_dict[target] = temp_df

    # else:
    #     stats_dict[target] = pd.DataFrame()

    # --- 확인용 출력 (루프 돌 때마다 확인 가능) ---
    print(f"[{target}]")
    print(f"  - 전체 데이터 수: {len(subset_df)}")
    print(f"  - Non-conflict (일관된 데이터): {len(non_conflict)}")
    print(f"  - Conflict (섞인 데이터, 처리 전): {len(conflict)}")
    print(f"  - Conflict (처리 후, stats_dict 저장): {len(stats_dict[target])}")
    print("-" * 30)

# stats_dict['Metastasis'] 에는 처리된 conflict 데이터만 들어있습니다.

In [ ]:
import pandas as pd

# 한 파일에 여러 시트로 저장 (Metastasis, Recurrence 모두 포함)
excel_path = "stats_terms.xlsx"

with pd.ExcelWriter(excel_path) as writer:
    for target in ['Metastasis', 'Recurrence']:
        temp_df = stats_dict[target].copy()
        pos_term_temp_df = temp_df.loc[temp_df['Terms'] >= 0].copy()

        # 수술일자, 검사접수일자 yyyy-mm-dd 형식으로 변환
        for col in ['수술일자', '검사접수일자']:
            temp_df[col] = pd.to_datetime(temp_df[col], errors='coerce').dt.strftime('%Y-%m-%d')
            pos_term_temp_df[col] = pd.to_datetime(pos_term_temp_df[col], errors='coerce').dt.strftime('%Y-%m-%d')

        temp_df.to_excel(writer, sheet_name=f'{target}_all', index=False)
        pos_term_temp_df.to_excel(writer, sheet_name=f'{target}_Terms_positive', index=False)


In [ ]:
merged_term = pd.merge(
    stats_dict['Metastasis'], 
    stats_dict['Recurrence'], 
    how = 'outer', 
    on = ['병원등록번호', '성별','암종','수술일자', '검사접수일자']
)


for col in ['수술일자', '검사접수일자']:
    merged_term[col] = pd.to_datetime(merged_term[col], errors='coerce').dt.strftime('%Y-%m-%d')


merged_term.to_excel('stats_terms_merged.xlsx', index=False)


In [ ]:
merged_term.groupby(['병원등록번호', '수술일자', '검사접수일자']).size().value_counts()

In [ ]:
merged_term.shape

In [ ]:
import matplotlib.pyplot as plt

target = 'Recurrence'
temp_df = stats_dict[target]
pos_term_temp_df = stats_dict[target].loc[stats_dict[target]['Terms'] >= 0]

fig, ax1 = plt.subplots(figsize=(8, 4))

btarget = 'b' + target
# 두 클래스 분리 (positive, negative)
vals_pos = pos_term_temp_df[pos_term_temp_df[btarget] == 'positive']['Terms'].dropna()
vals_neg = pos_term_temp_df[pos_term_temp_df[btarget] == 'negative']['Terms'].dropna()

# 두번째 y축(우): negative
ax2 = ax1.twinx()
sns.histplot(vals_neg, ax=ax2, color='tab:blue', alpha=0.5, label='negative', binwidth=5)
ax2.set_ylabel('Count (negative)', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')


# 첫번째 y축(좌): positive
sns.histplot(vals_pos, ax=ax1, color='tab:red', alpha=0.5, label='positive', binwidth=5)
ax1.set_ylabel('Count (positive)', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:red')
ax1.set_xlabel(r'$\Delta$day')


# 범례 수동 설정
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title(f"{target} Histogram - Overlapped")
fig.tight_layout()
fig.savefig(os.path.join(figdir, f'{target}_histogram_overlapping.png'), dpi=400)
# plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 스택형(stack) histogram을 그리기 위해 두 클래스를 합침
btarget = 'b' + target
vals_pos = pos_term_temp_df[pos_term_temp_df[btarget] == 'positive']['Terms'].dropna()
vals_neg = pos_term_temp_df[pos_term_temp_df[btarget] == 'negative']['Terms'].dropna()

import numpy as np
import seaborn as sns

# 데이터 프레임을 만들어서 합침
stack_data = [
    pd.DataFrame({'Terms': vals_pos, 'class': 'positive'}),
    pd.DataFrame({'Terms': vals_neg, 'class': 'negative'})
]
stack_df = pd.concat(stack_data, ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 4))

# 스택형 히스토그램 (stacked=True), y축 로그 스케일(log scale)
classes = ['positive', 'negative']
colors = ['tab:red', 'tab:blue']
hist = sns.histplot(
    data=stack_df, x='Terms', hue='class',
    hue_order=classes,
    # bins=30,
    binwidth = 5,
    multiple='stack',
    palette=dict(zip(classes, colors)),
    ax=ax,
    alpha=0.7,
    edgecolor='k'
)

ax.set_yscale('log')
ax.set_ylabel('Count - log scale')
ax.set_xlabel(r'$\Delta$day')

# get_legend_handles_labels가 안될 때 legend 직접 생성
ax.legend(handles=[
    plt.Line2D([], [], color='tab:red', lw=10, label='positive', solid_capstyle='butt', linewidth=10, alpha=0.7, marker=None, linestyle='-', dash_capstyle='butt'),
    plt.Line2D([], [], color='tab:blue', lw=10, label='negative', solid_capstyle='butt', linewidth=10, alpha=0.7, marker=None, linestyle='-', dash_capstyle='butt')
], title='class', labels=classes, handlelength=1.2)

plt.title(f"{target} Histogram - Stacked log scale")
fig.tight_layout()
fig.savefig(os.path.join(figdir, f'{target}_histogram_stacking.png'), dpi=400)